# H5AD Test Fixture Generator

Generates small H5AD files for unit testing the H5adAdapter.
Each file targets a specific edge case with minimal data (1-10 rows).

**Requirements**: `pip install anndata numpy scipy pandas`

In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import os

VALID_DIR = "valid"
EDGE_DIR = "edge_cases"
INVALID_DIR = "invalid"

os.makedirs(VALID_DIR, exist_ok=True)
os.makedirs(EDGE_DIR, exist_ok=True)
os.makedirs(INVALID_DIR, exist_ok=True)

print("Directories ready.")

Directories ready.


---
## Valid Cases
These should all load successfully.

### valid/dense_2d.h5ad
Minimal valid dataset: dense expression matrix, 2D spatial coords in obsm/X_spatial, genes in var/_index, one categorical cluster column.

In [2]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{VALID_DIR}/dense_2d.h5ad")
print(adata)

AnnData object with n_obs × n_vars = 5 × 3
    obs: 'leiden'
    obsm: 'X_spatial'


### valid/dense_3d.h5ad
Dense matrix with 3D spatial coordinates.

In [3]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "C", "A", "B"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 3).astype(np.float32)},
)
adata.write_h5ad(f"{VALID_DIR}/dense_3d.h5ad")
print(adata)

AnnData object with n_obs × n_vars = 5 × 3
    obs: 'leiden'
    obsm: 'X_spatial'


### valid/sparse_csr.h5ad
CSR sparse expression matrix. **Note: sparse reading is not yet implemented — this file is for future testing.**

In [4]:
n_cells, n_genes = 5, 3
dense = np.array([[1, 0, 0], [0, 2, 0], [0, 0, 3], [1, 1, 0], [0, 0, 0]], dtype=np.float32)
adata = ad.AnnData(
    X=sp.csr_matrix(dense),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "C", "A", "B"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{VALID_DIR}/sparse_csr.h5ad")
print(f"X type: {type(adata.X)}, shape: {adata.X.shape}")

X type: <class 'scipy.sparse._csr.csr_matrix'>, shape: (5, 3)


### valid/sparse_csc.h5ad
CSC sparse expression matrix. **Note: sparse reading is not yet implemented — this file is for future testing.**

In [5]:
n_cells, n_genes = 5, 3
dense = np.array([[1, 0, 0], [0, 2, 0], [0, 0, 3], [1, 1, 0], [0, 0, 0]], dtype=np.float32)
adata = ad.AnnData(
    X=sp.csc_matrix(dense),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "C", "A", "B"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{VALID_DIR}/sparse_csc.h5ad")
print(f"X type: {type(adata.X)}, shape: {adata.X.shape}")

X type: <class 'scipy.sparse._csc.csc_matrix'>, shape: (5, 3)


### valid/coords_in_obsm_spatial.h5ad
Coordinates in `obsm/spatial` (secondary key, not X_spatial).

In [6]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{VALID_DIR}/coords_in_obsm_spatial.h5ad")
print("obsm keys:", list(adata.obsm.keys()))

obsm keys: ['spatial']


### valid/coords_in_obs_columns.h5ad
Coordinates in obs columns (`center_x`, `center_y`) — no obsm at all.

In [7]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({
        "center_x": np.random.rand(n_cells).astype(np.float32) * 1000,
        "center_y": np.random.rand(n_cells).astype(np.float32) * 1000,
        "leiden": pd.Categorical(["A", "B", "A", "B", "A"]),
    }, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
)
adata.write_h5ad(f"{VALID_DIR}/coords_in_obs_columns.h5ad")
print("obs columns:", list(adata.obs.columns))

obs columns: ['center_x', 'center_y', 'leiden']


### valid/coords_in_obs_xy.h5ad
Coordinates in obs columns (`x`, `y`) — lowest priority obs fallback.

In [8]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({
        "x": np.random.rand(n_cells).astype(np.float32) * 500,
        "y": np.random.rand(n_cells).astype(np.float32) * 500,
        "leiden": pd.Categorical(["A", "B", "A", "B", "A"]),
    }, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
)
adata.write_h5ad(f"{VALID_DIR}/coords_in_obs_xy.h5ad")
print("obs columns:", list(adata.obs.columns))

obs columns: ['x', 'y', 'leiden']


### valid/with_embeddings.h5ad
Has X_spatial + X_umap + X_pca embeddings.

In [9]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "C", "A", "B"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={
        "X_spatial": np.random.rand(n_cells, 2).astype(np.float32),
        "X_umap": np.random.rand(n_cells, 2).astype(np.float32),
        "X_pca": np.random.rand(n_cells, 3).astype(np.float32),
    },
)
adata.write_h5ad(f"{VALID_DIR}/with_embeddings.h5ad")
print("obsm keys:", list(adata.obsm.keys()))

obsm keys: ['X_spatial', 'X_umap', 'X_pca']


### valid/multiple_cluster_columns.h5ad
Multiple obs columns: leiden (categorical), celltype (categorical), n_genes (numerical).

In [10]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({
        "leiden": pd.Categorical(["0", "1", "2", "0", "1"]),
        "celltype": pd.Categorical(["Neuron", "Astrocyte", "Neuron", "Microglia", "Astrocyte"]),
        "n_genes": [150, 200, 180, 220, 160],
        "total_counts": [3000.5, 4200.1, 3800.0, 5100.3, 2900.7],
    }, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{VALID_DIR}/multiple_cluster_columns.h5ad")
print(adata.obs)

       leiden   celltype  n_genes  total_counts
cell_0      0     Neuron      150        3000.5
cell_1      1  Astrocyte      200        4200.1
cell_2      2     Neuron      180        3800.0
cell_3      0  Microglia      220        5100.3
cell_4      1  Astrocyte      160        2900.7


### valid/scanpy_palette.h5ad
Has cluster colors stored in `uns/{column}_colors` (Scanpy convention).

In [11]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "C", "A", "B"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
    uns={"leiden_colors": np.array(["#ff0000", "#00ff00", "#0000ff"])},
)
adata.write_h5ad(f"{VALID_DIR}/scanpy_palette.h5ad")
print("uns keys:", list(adata.uns.keys()))

uns keys: ['leiden_colors']


### valid/genes_in_var_gene.h5ad
Gene names in `var/gene` column instead of `var/_index`.

In [12]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame({"gene": ["MyGene1", "MyGene2", "MyGene3"]}, index=["idx0", "idx1", "idx2"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{VALID_DIR}/genes_in_var_gene.h5ad")
print("var columns:", list(adata.var.columns))
print("var index:", list(adata.var.index))

var columns: ['gene']
var index: ['idx0', 'idx1', 'idx2']


---
## Edge Cases
These should load but with specific handling (warnings, fallbacks, partial data).

### edge_cases/no_genes.h5ad
No gene names available (empty var). Should load with empty gene list.

In [13]:
n_cells = 5
adata = ad.AnnData(
    X=np.zeros((n_cells, 0), dtype=np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/no_genes.h5ad")
print(f"genes: {adata.n_vars}, cells: {adata.n_obs}")

genes: 0, cells: 5


### edge_cases/no_clusters.h5ad
No obs columns (only spatial). Should load with no cluster data.

In [14]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame(index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/no_clusters.h5ad")
print(f"obs columns: {list(adata.obs.columns)}")

obs columns: []


### edge_cases/no_embeddings.h5ad
Only X_spatial in obsm, no UMAP/PCA. Should load with empty embeddings.

In [15]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/no_embeddings.h5ad")
print("obsm keys:", list(adata.obsm.keys()))

obsm keys: ['X_spatial']


### edge_cases/coords_with_nan.h5ad
Some coordinates have NaN values. Should drop invalid rows if >=90% are still valid.

In [16]:
n_cells, n_genes = 10, 3
coords = np.random.rand(n_cells, 2).astype(np.float32)
coords[0, 0] = np.nan  # 1 bad out of 10 = 90% valid, should pass

adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A"] * 5 + ["B"] * 5)}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": coords},
)
adata.write_h5ad(f"{EDGE_DIR}/coords_with_nan.h5ad")
print(f"NaN count: {np.isnan(coords).sum()}")

NaN count: 1


### edge_cases/coords_with_inf.h5ad
Some coordinates have Inf values.

In [17]:
n_cells, n_genes = 10, 3
coords = np.random.rand(n_cells, 2).astype(np.float32)
coords[0, 0] = np.inf

adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A"] * 5 + ["B"] * 5)}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": coords},
)
adata.write_h5ad(f"{EDGE_DIR}/coords_with_inf.h5ad")
print(f"Inf count: {np.isinf(coords).sum()}")

Inf count: 1


### edge_cases/coords_all_zeros.h5ad
All coordinates are zero.

In [18]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.zeros((n_cells, 2), dtype=np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/coords_all_zeros.h5ad")
print("All coords zero")

All coords zero


### edge_cases/z_mostly_zeros.h5ad
3D coordinates where Z is mostly zeros. Should fall back to 2D (>=90% of Z must be non-zero for 3D).

In [19]:
n_cells, n_genes = 10, 3
coords = np.random.rand(n_cells, 3).astype(np.float32)
coords[:, 2] = 0  # All Z = 0
coords[0, 2] = 1.0  # Only 1 non-zero Z = 10%, below 90% threshold

adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A"] * 5 + ["B"] * 5)}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": coords},
)
adata.write_h5ad(f"{EDGE_DIR}/z_mostly_zeros.h5ad")
print(f"Non-zero Z: {np.count_nonzero(coords[:, 2])}/{n_cells}")

Non-zero Z: 1/10


### edge_cases/expression_all_zeros.h5ad
Expression matrix is all zeros. Should load without error.

In [20]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.zeros((n_cells, n_genes), dtype=np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/expression_all_zeros.h5ad")
print(f"X sum: {adata.X.sum()}")

X sum: 0.0


### edge_cases/expression_with_nan.h5ad
Expression matrix has NaN values. Should treat NaN as 0 (non-fatal).

In [21]:
n_cells, n_genes = 5, 3
X = np.random.rand(n_cells, n_genes).astype(np.float32)
X[0, 0] = np.nan
X[2, 1] = np.nan

adata = ad.AnnData(
    X=X,
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/expression_with_nan.h5ad")
print(f"NaN count in X: {np.isnan(X).sum()}")

NaN count in X: 2


### edge_cases/cluster_single_value.h5ad
Cluster column with only one unique value.

In [22]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "A", "A", "A", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/cluster_single_value.h5ad")
print(f"Unique clusters: {adata.obs['leiden'].nunique()}")

Unique clusters: 1


### edge_cases/cluster_with_nulls.h5ad
Cluster column with some NaN/null values mixed in.

In [23]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({
        "celltype": pd.Categorical(["Neuron", None, "Astrocyte", None, "Neuron"]),
    }, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/cluster_with_nulls.h5ad")
print(adata.obs["celltype"])

cell_0       Neuron
cell_1          NaN
cell_2    Astrocyte
cell_3          NaN
cell_4       Neuron
Name: celltype, dtype: category
Categories (2, object): ['Astrocyte', 'Neuron']


### edge_cases/cluster_many_unique.h5ad
Cluster column with >100 unique values (should auto-detect as numerical).

In [24]:
n_cells, n_genes = 150, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({
        "cell_id_col": [f"id_{i}" for i in range(n_cells)],  # 150 unique strings
    }, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/cluster_many_unique.h5ad")
print(f"Unique values: {adata.obs['cell_id_col'].nunique()}")

Unique values: 150


### edge_cases/leiden_numerical_values.h5ad
Column named 'leiden' but with numerical values. Should be forced categorical by name.

In [25]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({
        "leiden": [0, 1, 2, 0, 1],  # Integers, not categorical
    }, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/leiden_numerical_values.h5ad")
print(f"leiden dtype: {adata.obs['leiden'].dtype}")

leiden dtype: int64


### edge_cases/single_cell.h5ad
Dataset with exactly 1 cell.

In [26]:
adata = ad.AnnData(
    X=np.array([[1.0, 2.0, 3.0]], dtype=np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A"])}, index=["cell_0"]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.array([[0.5, 0.3]], dtype=np.float32)},
)
adata.write_h5ad(f"{EDGE_DIR}/single_cell.h5ad")
print(f"cells: {adata.n_obs}")

cells: 1


### edge_cases/scanpy_palette_mismatch.h5ad
Palette in uns has wrong number of colors (doesn't match cluster count). Should fall back to default palette.

In [27]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "C", "A", "B"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
    uns={"leiden_colors": np.array(["#ff0000", "#00ff00"])},  # 2 colors for 3 clusters
)
adata.write_h5ad(f"{EDGE_DIR}/scanpy_palette_mismatch.h5ad")
print(f"Clusters: {adata.obs['leiden'].nunique()}, palette colors: {len(adata.uns['leiden_colors'])}")

Clusters: 3, palette colors: 2


---
## Invalid Cases
These should fail with clear error messages.

### invalid/no_coordinates.h5ad
No spatial coordinates anywhere (no obsm, no obs xy columns). Should throw error.

In [28]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
)
adata.write_h5ad(f"{INVALID_DIR}/no_coordinates.h5ad")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"obs columns: {list(adata.obs.columns)}")

obsm keys: []
obs columns: ['leiden']


### invalid/coords_with_strings.h5ad
Coordinate obs columns contain string values. Should fail validation (<90% valid).

In [29]:
n_cells, n_genes = 5, 3
adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({
        "center_x": ["abc", "def", "ghi", "jkl", "mno"],
        "center_y": ["abc", "def", "ghi", "jkl", "mno"],
        "leiden": pd.Categorical(["A", "B", "A", "B", "A"]),
    }, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
)
adata.write_h5ad(f"{INVALID_DIR}/coords_with_strings.h5ad")
print(f"center_x dtype: {adata.obs['center_x'].dtype}")

center_x dtype: object


### invalid/coords_mostly_nan.h5ad
More than 10% of coordinates are NaN. Should fail the >=90% validation.

In [30]:
n_cells, n_genes = 10, 3
coords = np.random.rand(n_cells, 2).astype(np.float32)
coords[:3, 0] = np.nan  # 3 out of 10 = 70% valid, below 90%

adata = ad.AnnData(
    X=np.random.rand(n_cells, n_genes).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A"] * 5 + ["B"] * 5)}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=["GeneA", "GeneB", "GeneC"]),
    obsm={"X_spatial": coords},
)
adata.write_h5ad(f"{INVALID_DIR}/coords_mostly_nan.h5ad")
print(f"Valid coords: {(~np.isnan(coords[:, 0])).sum()}/{n_cells}")

Valid coords: 7/10


### invalid/matrix_shape_mismatch.h5ad
Expression matrix has different number of genes than var. Should cause issues during gene queries.

In [31]:
# Note: AnnData enforces shape consistency, so we create a valid file
# then manually edit it to break the relationship.
# For now, create a file where var has extra columns that might confuse the adapter.
n_cells = 5
adata = ad.AnnData(
    X=np.random.rand(n_cells, 3).astype(np.float32),
    obs=pd.DataFrame({"leiden": pd.Categorical(["A", "B", "A", "B", "A"])}, index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame({"gene": ["Gene1", "Gene2", "Gene3"], "extra": ["x", "y", "z"]}, index=["a", "b", "c"]),
    obsm={"X_spatial": np.random.rand(n_cells, 2).astype(np.float32)},
)
# Note: True shape mismatch is hard to create with anndata — it validates.
# This test at least verifies the adapter handles var with extra columns.
adata.write_h5ad(f"{INVALID_DIR}/matrix_shape_mismatch.h5ad")
print(f"X shape: {adata.X.shape}, var shape: {adata.var.shape}")

X shape: (5, 3), var shape: (3, 2)


---
## Summary

### Checklist

**Valid (should load successfully):**
- [ ] `valid/dense_2d.h5ad` — Dense matrix, 2D obsm/X_spatial
- [ ] `valid/dense_3d.h5ad` — Dense matrix, 3D obsm/X_spatial
- [ ] `valid/sparse_csr.h5ad` — CSR sparse matrix (future: not yet supported)
- [ ] `valid/sparse_csc.h5ad` — CSC sparse matrix (future: not yet supported)
- [ ] `valid/coords_in_obsm_spatial.h5ad` — Coords in obsm/spatial (secondary key)
- [ ] `valid/coords_in_obs_columns.h5ad` — Coords in obs center_x/center_y
- [ ] `valid/coords_in_obs_xy.h5ad` — Coords in obs x/y (lowest priority)
- [ ] `valid/with_embeddings.h5ad` — X_spatial + X_umap + X_pca
- [ ] `valid/multiple_cluster_columns.h5ad` — leiden + celltype + n_genes + total_counts
- [ ] `valid/scanpy_palette.h5ad` — Palette from uns/leiden_colors
- [ ] `valid/genes_in_var_gene.h5ad` — Gene names in var/gene column

**Edge Cases (should load with warnings/fallbacks):**
- [ ] `edge_cases/no_genes.h5ad` — Empty var → empty gene list
- [ ] `edge_cases/no_clusters.h5ad` — No obs columns → no clusters
- [ ] `edge_cases/no_embeddings.h5ad` — Only X_spatial → empty embeddings
- [ ] `edge_cases/coords_with_nan.h5ad` — 1/10 NaN coords → drops row, still loads
- [ ] `edge_cases/coords_with_inf.h5ad` — Inf in coords
- [ ] `edge_cases/coords_all_zeros.h5ad` — All zero coords
- [ ] `edge_cases/z_mostly_zeros.h5ad` — 3D coords but Z mostly zero → falls back to 2D
- [ ] `edge_cases/expression_all_zeros.h5ad` — All-zero expression matrix
- [ ] `edge_cases/expression_with_nan.h5ad` — NaN in expression → treat as 0
- [ ] `edge_cases/cluster_single_value.h5ad` — Only one unique cluster value
- [ ] `edge_cases/cluster_with_nulls.h5ad` — Null values in cluster column
- [ ] `edge_cases/cluster_many_unique.h5ad` — >100 unique values → auto-detect as numerical
- [ ] `edge_cases/leiden_numerical_values.h5ad` — leiden with integers → forced categorical by name
- [ ] `edge_cases/single_cell.h5ad` — Exactly 1 cell
- [ ] `edge_cases/scanpy_palette_mismatch.h5ad` — Wrong palette count → fallback to default

**Invalid (should fail with clear errors):**
- [ ] `invalid/no_coordinates.h5ad` — No spatial data anywhere → error
- [ ] `invalid/coords_with_strings.h5ad` — String coordinates → validation error
- [ ] `invalid/coords_mostly_nan.h5ad` — >10% NaN → validation error
- [ ] `invalid/matrix_shape_mismatch.h5ad` — Var has extra columns

---
## Run All
Run this cell to generate all fixtures at once.

In [32]:
import glob

files = sorted(glob.glob("**/*.h5ad", recursive=True))
print(f"Generated {len(files)} test fixtures:")
for f in files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

Generated 30 test fixtures:
  edge_cases/cluster_many_unique.h5ad (28.5 KB)
  edge_cases/cluster_single_value.h5ad (19.5 KB)
  edge_cases/cluster_with_nulls.h5ad (19.5 KB)
  edge_cases/coords_all_zeros.h5ad (19.5 KB)
  edge_cases/coords_with_inf.h5ad (19.5 KB)
  edge_cases/coords_with_nan.h5ad (19.5 KB)
  edge_cases/expression_all_zeros.h5ad (19.5 KB)
  edge_cases/expression_with_nan.h5ad (19.5 KB)
  edge_cases/leiden_numerical_values.h5ad (17.8 KB)
  edge_cases/no_clusters.h5ad (17.5 KB)
  edge_cases/no_embeddings.h5ad (19.5 KB)
  edge_cases/no_genes.h5ad (19.5 KB)
  edge_cases/scanpy_palette_mismatch.h5ad (20.0 KB)
  edge_cases/single_cell.h5ad (19.5 KB)
  edge_cases/z_mostly_zeros.h5ad (19.5 KB)
  invalid/coords_mostly_nan.h5ad (19.5 KB)
  invalid/coords_with_strings.h5ad (19.4 KB)
  invalid/matrix_shape_mismatch.h5ad (20.1 KB)
  invalid/no_coordinates.h5ad (18.8 KB)
  valid/coords_in_obs_columns.h5ad (19.4 KB)
  valid/coords_in_obs_xy.h5ad (19.4 KB)
  valid/coords_in_obsm_spatial.h